# Exploring comparing ToponymExtractor outputs to GB 1900 Gazetteer labels

Overall this notebook shows that there are very few "perfect" predictions

In [ ]:
from typing import Final
from pathlib import Path
from os import getenv
from dotenv import find_dotenv, load_dotenv
import re
import pandas as pd
import geopandas as gp
import matplotlib.pyplot as plt

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

In [ ]:
# Load GB1900 gazetteer
gb1900 = gp\
    .read_file(LOCAL_DIR.joinpath("outputs/pngs/text-locations.gpkg"))
gb1900 = gb1900[["pin_id", "tiff_filename", "final_text", "geometry"]]
gb1900 = gb1900.drop_duplicates()
gb1900["final_text"] = gb1900.final_text.str.replace(r"\s*-\s*", " ", regex = True)
gb1900["final_text"] = gb1900.final_text.str.replace(r"\s+", " ", regex = True)
gb1900["final_text"] = gb1900.final_text.str.lower()
gb1900

In [ ]:
pred_dir = LOCAL_DIR\
    .joinpath("outputs/toponym-extractor-ambiguous-masks/Refined")
pred_fp_list = [*pred_dir.glob("*.gpkg")]

In [ ]:
example_idx = 20
pred_fp_eg = pred_fp_list[example_idx]
pred_fp_eg

In [ ]:
mask_preds = gp.read_file(pred_fp_eg)
mask_preds = mask_preds.sort_values(["png_filename", "groupid", "wordid"])
mask_preds

In [ ]:
# Normalize groupid
mask_preds["key"] =\
    mask_preds.png_filename + mask_preds.groupid.astype("string")
key_group = mask_preds[["key"]].drop_duplicates(ignore_index = True)
key_group["groupid"] = [*range(len(key_group))]
mask_preds = pd.merge(
    mask_preds[["png_filename", "wordid", "word", "score", "geometry", "key"]],
    key_group,
    on = "key"
)
mask_preds

In [ ]:
mask_preds = gp.read_file(pred_fp_eg)
mask_preds = mask_preds.sort_values(["png_filename", "groupid", "wordid"])

toponyms = mask_preds[["png_filename", "groupid", "word"]]\
    .groupby(["png_filename", "groupid"], as_index = False)\
    .agg(words = ("word", " ".join))
toponyms["key"] = toponyms.png_filename + toponyms.groupid.astype("string")

toponym_polygons = mask_preds[["png_filename", "groupid", "geometry"]]\
    .dissolve(by = ["png_filename", "groupid"], as_index = True)\
    .convex_hull\
    .reset_index(drop = False, name = "geometry")
toponym_polygons["key"] =\
    toponym_polygons.png_filename + toponym_polygons.groupid.astype("string")

toponyms = pd.merge(toponym_polygons, toponyms[["key", "words"]], on = "key")
toponyms = toponyms.drop(columns = ["key"])
toponyms["words"] = toponyms.words.str.replace(r"\s*-\s*", " ", regex = True)
toponyms["words"] = toponyms.words.str.replace(r"\s+", " ", regex = True)
toponyms["words"] = toponyms.words.str.lower()
toponyms = toponyms[["png_filename", "groupid", "words", "geometry"]]

toponyms["geometry"] = toponyms.geometry.buffer(10)

toponyms = gp.sjoin(
    toponyms,
    gb1900[["final_text", "geometry"]].drop_duplicates(),
    how = "left"
)

toponyms

In [ ]:
(toponyms.words == toponyms.final_text).sum()